In [2]:
import numpy as np
import random
import scipy.special as sc_special

random.seed(42)
np.random.seed(42)

days, initial_price = 100, 100
prices = initial_price * np.cumprod(1 + np.random.normal(0.001, 0.02, days))

NEST_COUNT, GENERATIONS = 20, 10
PA = 0.25
TRANSACTION_FEE, DRAWDOWN_PENALTY = 0.001, 0.5


def evaluate(strategy):
    short, long = sorted([int(strategy[0]), int(strategy[1])])
    short = max(2, min(15, short))
    long = max(16, min(40, long))

    s_ma = np.convolve(prices, np.ones(short) / short, mode="full")[:days]
    l_ma = np.convolve(prices, np.ones(long) / long, mode="full")[:days]

    position = np.roll((s_ma > l_ma).astype(float), 1)
    position[0] = 0
    trades = np.abs(np.diff(position, prepend=0))

    mkt_ret = np.zeros(days)
    mkt_ret[1:] = prices[1:] / prices[:-1] - 1
    strat_ret = position * mkt_ret - trades * TRANSACTION_FEE

    portfolio = np.cumprod(1 + strat_ret)
    net_profit = (portfolio[-1] - 1) * 100
    peak = np.maximum.accumulate(portfolio)
    max_drawdown = np.max((peak - portfolio) / peak) * 100
    total_fees = np.sum(trades * TRANSACTION_FEE) * 100

    fitness = net_profit - DRAWDOWN_PENALTY * max_drawdown
    return fitness, net_profit, max_drawdown, total_fees, [short, long]


def levy_flight(beta=1.5):
    sigma = (
        sc_special.gamma(1 + beta)
        * np.sin(np.pi * beta / 2)
        / (sc_special.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2)))
    ) ** (1 / beta)
    u = np.random.normal(0, sigma, size=2)
    v = np.random.normal(0, 1, size=2)
    return u / (np.abs(v) ** (1 / beta))


nests = np.array([
    [random.randint(2, 15), random.randint(16, 40)] for _ in range(NEST_COUNT)
], dtype=float)

fitnesses = [evaluate(n)[0] for n in nests]

best_idx = np.argmax(fitnesses)
best_strat = evaluate(nests[best_idx])[4]
best_fit = fitnesses[best_idx]

for _ in range(GENERATIONS):
    for i in range(NEST_COUNT):
        step = levy_flight()
        new_nest = nests[i] + 0.01 * step * (nests[i] - nests[best_idx])
        new_fit, _, _, _, new_clean_nest = evaluate(new_nest)

        j = random.randint(0, NEST_COUNT - 1)
        if new_fit > fitnesses[j]:
            nests[j] = new_clean_nest
            fitnesses[j] = new_fit

    sorted_indices = np.argsort(fitnesses)
    abandon_count = int(PA * NEST_COUNT)
    for k in sorted_indices[:abandon_count]:
        nests[k] = [random.randint(2, 15), random.randint(16, 40)]
        fitnesses[k] = evaluate(nests[k])[0]

    current_best_idx = np.argmax(fitnesses)
    if fitnesses[current_best_idx] > best_fit:
        best_fit = fitnesses[current_best_idx]
        best_strat = evaluate(nests[current_best_idx])[4]

fit, profit, dd, fees, clean_strat = evaluate(best_strat)

print(f"Best Strategy: Short={clean_strat[0]}, Long={clean_strat[1]}")
print(
    f"Profit: {profit:.2f}% | Drawdown: {dd:.2f}% | Fees: {fees:.2f}% | Fitness:"
    f" {fit:.2f}"
)

Best Strategy: Short=11, Long=22
Profit: 4.88% | Drawdown: 13.95% | Fees: 0.50% | Fitness: -2.10
